# DS340W - HIWL rerun with the saved split
Retrains HIWL (EfficientNet-B1) on the saved split (`splits/train_set.csv`, `test_set.csv`, `validation_unseen_set.csv`), then reruns the uncertainty layer on the unseen set.

Settings: GPU T4 x2, Internet on, Input = the *DS340W* notebook output (for `clean_gzdata5`).

In [ ]:
!git clone -q https://github.com/asamad0687/HIWL-DS340W.git /kaggle/working/HIWL-DS340W
%cd /kaggle/working/HIWL-DS340W/scheme
!grep -c HIWL_SPLIT_DIR utils.py
!wc -l ../splits/*.csv

In [ ]:
import os, glob
DATA = [p for p in glob.glob('/kaggle/input/**/clean_gzdata5', recursive=True) if os.path.isdir(p)][0]
os.environ['HIWL_SPLIT_DIR'] = '/kaggle/working/HIWL-DS340W/splits'
print(DATA)
# quick check that the saved split loads and every image exists
from utils import read_split_data
_ = read_split_data(DATA)

In [ ]:
# Train HIWL on the saved split (same settings as the reproduction run, about 8.5 hours)
!python train_efficientnet.py --data-path5 "$DATA" --weights4 "" --weights2 "" --os linux --device cuda:0 --num-model B1

In [ ]:
# Per-class metrics on the unseen set
!python evaluate_metrics.py --data-path5 "$DATA" --num-model B1

In [ ]:
# HIWL picks its final model pair on the val set (acc_combine_best_index in the log).
# Find the saved weight files that reproduce HIWL's own test predictions, and use those.
import sys, shutil, torch, torch.nn.functional as F
from torchvision import transforms
sys.path.insert(0, '/kaggle/working/HIWL-DS340W/novel')
import hiwl_uncertainty as H
from my_dataset import MyDataSet
dev = torch.device('cuda:0')
_, _, _, _, tp, tl = read_split_data(DATA)
tf = transforms.Compose([transforms.CenterCrop(240), transforms.ToTensor(),
                         transforms.Normalize([0.046, 0.041, 0.030], [0.090, 0.075, 0.065])])
dl = torch.utils.data.DataLoader(MyDataSet(tp, tl, transform=tf), batch_size=64, shuffle=False,
                                 num_workers=2, collate_fn=MyDataSet.collate_fn)
ref = torch.load('predicts/B1/test_pred_all.pth', map_location='cpu').long()

@torch.no_grad()
def probs(n, path):
    m = H.load(n, path, dev)
    return torch.cat([F.softmax(m(x.to(dev)), 1).cpu() for x, _ in dl])

P4 = {j: probs(4, f'weights/B1/best_model4_{j}.pth') for j in range(3) if os.path.exists(f'weights/B1/best_model4_{j}.pth')}
P2 = {i: probs(2, f'weights/B1/best_model2_{i}.pth') for i in range(3) if os.path.exists(f'weights/B1/best_model2_{i}.pth')}
agree = {(i, j): (H.hierarchical_predict(P4[j], P2[i]) == ref).float().mean().item() for i in P2 for j in P4}
for k, v in sorted(agree.items()): print('model2', k[0], 'model4', k[1], 'agreement', round(v, 4))
i, j = max(agree, key=agree.get)
os.makedirs('/kaggle/working/best_pair', exist_ok=True)
shutil.copy(f'weights/B1/best_model4_{j}.pth', '/kaggle/working/best_pair/best_model4_0.pth')
shutil.copy(f'weights/B1/best_model2_{i}.pth', '/kaggle/working/best_pair/best_model2_0.pth')
print('Using model2 file', i, '+ model4 file', j, '-> agreement', round(agree[(i, j)], 4))
!grep acc_combine_best_index log/efficientnet-B1

In [ ]:
# Novel approach: uncertainty layer on the unseen set
!python ../novel/hiwl_uncertainty.py --data-path5 "$DATA" --weights-dir /kaggle/working/best_pair --ref-pred predicts/B1/test_pred_all.pth --out-dir /kaggle/working/results_novel

In [ ]:
import pandas as pd
print(open('/kaggle/working/results_novel/summary.json').read())
pd.read_csv('/kaggle/working/results_novel/comparison_table.csv')

In [ ]:
!cd /kaggle/working && zip -qr rerun_results.zip results_novel HIWL-DS340W/scheme/log HIWL-DS340W/scheme/predicts HIWL-DS340W/scheme/results 2>/dev/null; ls -la /kaggle/working